# 5. Fine-Tuning a Chemical Language Model (CLM)

In this notebook, you will fine-tune a **pre-trained Chemical Language Model (CLM)** on your augmented molecular dataset.

**Prerequisites:**  
You have already run the augmentation notebook, which produced:
- `train.csv` — augmented training SMILES
- `val.csv` — augmented validation SMILES

**What is fine-tuning?**  
A CLM is first pre-trained on a large library of molecules, learning the general "grammar" of SMILES strings. Fine-tuning then adapts this pre-trained model to your smaller dataset, so that the model learns to generate molecules similar to your target set.

---

**Notebook structure:**
1. Imports & setup
2. Load the pre-training configuration
3. Read in the augmented data
4. Encode the SMILES
5. Prepare input/target sequences
6. Fine-tune the model
7. Sample new molecules
8. ✏️ Exercises

---
## Step 1 — Imports & Setup

We import:
- Standard libraries: `os`, `json`, `numpy`, `pandas`
- `tensorflow` / `keras` — the deep learning framework used to build and train the CLM
- Our own modules: `DataEncoding`, `CLM`, `split_input_target`, and `SamplingMolecules`

Make sure all provided module files (`encoding.py`, `model.py`, `sampling.py`) are in the **same folder** as this notebook.

In [ ]:
import os
import glob
import json
import numpy as np
import tensorflow as tf

# Our provided modules — make sure these .py files are in the same folder as this notebook
from evaluation import load_smiles
from evaluation.visualization import plot_training_history
from scripts.encoding import (
    DataEncoding,
)  # Handles tokenisation, padding, one-hot encoding
from scripts.model import (
    CLM,
    split_input_target,
)  # The CLM class and input/target splitter
from scripts.sampling import (
    SamplingMolecules,
)  # Handles molecule sampling after fine-tuning

# Fix random seed for reproducibility
tf.random.set_seed(42)

print("All imports successful!")

In [ ]:
# Automatically detect the directory where this notebook is located
base_dir = os.path.dirname(os.path.abspath("__file__"))

### Step 1b — Set paths

We define three directories:
- `results_dir` — where pre-training results are stored (model weights, config, vocabulary)
- `augmentation_dir` — where your augmented `train.csv` and `val.csv` are located (produced by the augmentation notebook)
- `saving_dir` — where fine-tuning outputs (model, history, sampled molecules) will be saved

In [ ]:
# Where pre-training results live (must contain best_combination.json,
# segment2label.json, and the pre-trained model.h5)
results_dir = os.path.join(base_dir, "results")

# Where your augmented CSVs are (output of notebook 4)
augmentation_dir = os.path.join(base_dir, "your_path_here")

# Where this notebook will save the fine-tuned model and sampled molecules
saving_dir = os.path.join(base_dir, "results", "finetuning")

# Create saving directory if it does not exist yet
if not os.path.exists(saving_dir):
    os.makedirs(saving_dir)
    print(f"Created saving directory: {saving_dir}")
else:
    print(f"Saving directory: {saving_dir}")

---
## Step 2 — Load the Pre-Training Configuration

The pre-trained model comes with two important files:

- **`combination.json`** — stores the hyperparameters (number of LSTM layers, hidden units, learning rate, etc.) used during pre-training. We reuse these for fine-tuning so the model architecture matches exactly.
- **`segment2label.json`** — the vocabulary dictionary: maps each SMILES token (e.g. `C`, `=`, `(`, `Br`) to an integer index. This must be identical to what the pre-trained model was built with — if the vocabulary changes, the model cannot be reused.

In [ ]:
# Load the best hyperparameter configuration from pre-training
pretraining_dir = os.path.join(results_dir, "pretraining")
config_path = os.path.join(pretraining_dir, "combination.json")

try:
    with open(config_path, "r") as f:
        best_config = json.load(f)  # Load as a Python dictionary
    print("Loaded combination.json successfully.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {config_path}."
        "Please make sure pre-training has been completed and the file exists."
    )

# Extract the hyperparameter sub-dictionary (architecture + training settings)
hp_combination = best_config["hps"]

# Load the vocabulary: maps each SMILES token to its integer index
vocab_path = os.path.join(results_dir, "segment2label.json")

try:
    with open(vocab_path, "r") as f:
        segment2label = json.load(f)
    print(f"Loaded vocabulary with {len(segment2label)} tokens.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {vocab_path}.\n"
        "This file should have been created during pre-training."
    )

# Quick peek at the loaded hyperparameters
print("Model hyperparameters:")
for key, value in hp_combination.items():
    print(f" {key}: {value}")

---
## Step 3 — Read in the Augmented Data

We load the `train.csv` and `val.csv` files produced by the augmentation notebook.
Each file has a `smiles` column containing the augmented SMILES strings.

No further splitting or augmentation is needed here — that was already handled.

We don't need to load `train.csv` yet, these molecules will be used during evaluation in the next notebook.

In [ ]:
# Read the augmented training set and extract the SMILES strings
augmented_train = load_smiles(os.path.join(augmentation_dir, "train.csv"))

# Read the augmented validation set and extract the SMILES strings
augmented_val = load_smiles(os.path.join(augmentation_dir, "val.csv"))

print(f"Training SMILES (augmented): {len(augmented_train)}")
print(f"Validation SMILES (augmented): {len(augmented_val)}")

# Quick look at the training data to confirm it loaded correctly
augmented_train[:10]

---
## Step 4 — Encode the SMILES

The model cannot work with raw SMILES strings — it needs numbers. The `DataEncoding` class handles three steps:

1. **Tokenisation** — split each SMILES into its chemical tokens (e.g. `CCO` → `['C', 'C', 'O']`)
2. **Add tokens** — prepend a start token (`G`) and an end token (`E`), then pad all sequences to the same length
3. **One-hot encoding** — convert each token into a binary vector of length `vocab_size`, using the pre-training vocabulary

The result is a 3D array of shape `(n_molecules, sequence_length, vocab_size)`.

In [ ]:
# Instantiate the encoding helper
encoding = DataEncoding()

# ── Training set ─────────────────────────────────────────────────────────────

# Step 1: Tokenise — split each SMILES string into individual chemical tokens
tokenized_train = encoding.tokenizer(augmented_train)

# Step 2: Add start ('G') and stop ('E') tokens, then pad to equal length
tokenized_train = encoding.add_tokens(
    tokenized_train, max_length=hp_combination["maxlen"]
)

# Step 3: One-hot encode using the pre-training vocabulary (segment2label)
# Each token becomes a binary vector; the vocabulary must match the pre-trained model's
one_hot_train, _ = encoding.one_hot_encoding(tokenized_train, segment2label)

# ── Validation set ────────────────────────────────────────────────────────────

# Same three steps, applied to the validation set
tokenized_val = encoding.tokenizer(augmented_val)
tokenized_val = encoding.add_tokens(tokenized_val, max_length=hp_combination["maxlen"])
one_hot_val, _ = encoding.one_hot_encoding(tokenized_val, segment2label)

print("Encoding complete.")
print(f"Training array shape: {one_hot_train.shape}")  # (n_train, seq_len, vocab_size)
print(f"Validation array shape: {one_hot_val.shape}")  # (n_val,   seq_len, vocab_size)

---
## Step 5 — Prepare Input / Target Sequences

A language model is trained to **predict the next token** given all previous tokens.

The `split_input_target` function creates this one-step shift:
- **Input (`X`)**: the sequence with the **last** token removed → `[G, C, C, O]`
- **Target (`y`)**: the sequence with the **first** token removed → `[C, C, O, E]`

At each position, the model sees all tokens so far and must predict the next one.
Since we use enumeration (not masking), there is no separate target sequence — we pass `None`.

In [ ]:
# Split each sequence into input and target
# sequence_target=None because enumeration does not use a masked target sequence
xTrain, yTrain = split_input_target(one_hot_train, sequence_target=None)
xVal, yVal = split_input_target(one_hot_val, sequence_target=None)

print("Input/target split complete.")
print(f"xTrain: {xTrain.shape}")  # (n_train, seq_len - 1, vocab_size)
print(f"yTrain: {yTrain.shape}")  # (n_train, seq_len - 1, vocab_size)
print(f"xVal:   {xVal.shape}")
print(f"yVal:   {yVal.shape}")

---
## Step 6 — Fine-Tune the Pre-Trained Model

We instantiate the `CLM` class in **`'Finetune'`** mode and call `fine_tune_model()`.

Key things happening inside `fine_tune_model()`:
- The **architecture** is rebuilt using the pre-training hyperparameters (same LSTM layers, same hidden units)
- The **weights** from the pre-trained model are loaded — the model already "knows" SMILES grammar
- The **learning rate** is automatically reduced to `learning_rate × 0.001` to avoid overwriting what was learned
- **Early stopping** halts training if validation loss stops improving (patience = 10 epochs)
- The **best checkpoint** is saved automatically via `ModelCheckpoint`

In [ ]:
# Small values (2–8) are typical for fine-tuning on (very) small datasets. In addition,
# reducing the learning rate by a factor of 100–1000 is often recommended to prevent catastrophic forgetting.
batch_size_finetune = 8  # Number of sequences per gradient update.
learning_rate_finetune = (
    hp_combination["learning_rate"] * 0.01
)  # Reduce by factor of 1000

# Inject the chosen batch size into the hyperparameter dictionary
# The CLM class will read it from here
hp_combination["batch_size_finetune"] = batch_size_finetune
hp_combination["learning_rate_finetune"] = learning_rate_finetune

print(f"Batch size: {batch_size_finetune}")
print(f"Learning rate (reduced): {learning_rate_finetune}")

In [ ]:
print(hp_combination)
# Instantiate the CLM in Finetune mode
trainer = CLM(
    model_parameters=hp_combination,            # Architecture + settings from pre-training
    mode="Finetune",                            # Activates fine-tuning logic inside the class
    saving_path=saving_dir,                     # Where to save checkpoints and the final model
    pre_trained_model_path=pretraining_dir,     # Folder containing the pre-trained model.h5)
)

In [ ]:
# Run fine-tuning

# Define the number of epochs to use for fine-tuning
trainer.epochs = 0

# Returns the trained Keras model object and the training history (loss per epoch)
model, history = trainer.fine_tune_model(
    xTrain=xTrain, yTrain=yTrain, xVal=xVal, yVal=yVal
)

print("Fine-tuning complete!")

In [ ]:
# Summarise the training run
val_losses = history.history["val_loss"]  # Validation loss recorded at every epoch

best_val_loss = min(val_losses)  # Lowest validation loss achieved
best_epoch = int(np.argmin(val_losses))  # Epoch at which it occurred (0-indexed)
stopping_epoch = len(val_losses)  # Last epoch run (early stopping may have cut this short)
final_val_loss = val_losses[-1]  # Validation loss at the very last epoch

# Save the fine-tuning summary alongside the pre-training config
best_config["fine_tuning"] = {
    "learning_rate_finetune": hp_combination["learning_rate"]
    * 0.001,  # Reduced LR actually used
    "batch_size_finetune": batch_size_finetune,
    "val_loss": final_val_loss,
    "stopping_epoch": stopping_epoch,
}

with open(os.path.join(saving_dir, "combination.json"), "w") as f:
    json.dump(best_config, f, indent=4)

print(f"Saved summary to: {os.path.join(saving_dir, 'best_combination.json')}")
print(f"Best validation loss: {best_val_loss:.4f} (epoch {best_epoch + 1})")
print(f"Final validation loss: {final_val_loss:.4f} (after {stopping_epoch} epochs)")

In [ ]:
# Save the training history as a json file
with open(os.path.join(saving_dir, "training_history.json"), "w") as f:
    json.dump(history.history, f, indent=4)

# Plot the training and validation loss & accuracy curves
fig = plot_training_history(history)

---
## Step 7 — Sample New Molecules

Now that the model is fine-tuned, we use it to **generate new molecules**.

The model generates SMILES one token at a time, starting from the start token `G`, until it produces the stop token `E` or reaches a maximum length.

**Temperature** controls how creative the sampling is:
- `T < 1.0` → conservative — sticks closer to what it learned (higher validity, lower diversity)
- `T = 1.0` → neutral
- `T > 1.0` → creative — more diverse but potentially less valid SMILES

We run **3 independent sampling rounds** to get a more robust picture of the model's output.

In [ ]:
n_samples = 1000  # Number of molecules to generate per round
temperature = 1.0  # Sampling temperature — see description above

print(f"Sampling {n_samples} molecules at temperature {temperature}.")

In [ ]:
# Load the fine-tuning config that was saved in the previous step
with open(os.path.join(saving_dir, "combination.json"), "r") as f:
    hp_space = json.load(f)

sampled_path = os.path.join(saving_dir, "unclean_sampled_molecules.smi")

# Instantiate the sampler — it loads the fine-tuned model from saving_dir
sampler = SamplingMolecules(
    sampling_parameter=hp_space,  # Contains path, architecture
    segment2label=segment2label,  # Vocabulary mapping tokens to integers
    saving_dir=saving_dir,
)

In [ ]:
n_samples = 10

# Generate molecules one token at a time, starting from the 'G' token
molecules = sampler.sample_multiple(
    num_sample=n_samples,
    temperature=temperature,
    starting_char="G",  # 'G' is the start-of-sequence token
)

# Save raw (unfiltered) output as JSON for later processing
with open(sampled_path, "w") as f:
    [f.write(f"{smi}\n") for smi in molecules]

print(f"Saved to: {sampled_path}")

---
## Step 8 — Sample from Checkpoint Models (Intermediate Epochs)

During fine-tuning, the model was saved at **every epoch** in the `all-epochs/` subdirectory. These checkpoint models capture the model's state at different stages of training.

**Why explore checkpoints?**
- Early checkpoints may have **higher diversity** but **lower validity** (model is still learning)
- Late checkpoints may have **higher validity** but **lower diversity** (model is overfitting to the training set)
- Finding the "sweet spot" can reveal the model's learning dynamics

We can load any checkpoint and sample molecules from it, just like we did with the final model.

In [ ]:
# Example: Sample from a specific checkpoint (e.g., epoch 10)

# List all saved checkpoints
checkpoints_dir = os.path.join(saving_dir, "all-epochs")
checkpoint_files = sorted(glob.glob(os.path.join(checkpoints_dir, "model-*.keras")))

print(f"Found {len(checkpoint_files)} checkpoints")
print(f"First checkpoint: {os.path.basename(checkpoint_files[0])}")
print(f"Last checkpoint: {os.path.basename(checkpoint_files[-1])}")

# Example: Sample from epoch 5
epoch_idx = 4  # 0-indexed, so this is epoch 5
checkpoint_path = checkpoint_files[epoch_idx]
checkpoint_epoch = int(os.path.basename(checkpoint_path).replace("model-", "").replace(".keras", ""))

print(f"\nLoading checkpoint from epoch {checkpoint_epoch}...")

# Create a sampler that uses this checkpoint instead of the final model
sampler_checkpoint = SamplingMolecules(
    sampling_parameter=hp_space,
    segment2label=segment2label,
    saving_dir=checkpoint_path,
)

# Sample molecules from this checkpoint
molecules_from_checkpoint = sampler_checkpoint.sample_multiple(
    num_sample=10,
    temperature=1.0,
    starting_char="G",
)

print(f"\nSampled from epoch {checkpoint_epoch}.")

sampled_path = os.path.join(saving_dir, f"unclean_sampled_molecules_epoch{epoch_idx+1}.smi")
with open(sampled_path, "w") as f:
    [f.write(f"{smi}\n") for smi in molecules_from_checkpoint]

print(f"Saved to {sampled_path}")


---
## Exercise 1 — Modify the Sampling

### Change the sampling temperature
Go back to **Step 7** and change `temperature` to `0.7`, then to `1.3`.  Then, procede to the evaluation: see notebook 6.
- Does a lower temperature give you **more valid** molecules?
- Does a higher temperature give you **more diverse** molecules?

## Exercise 2 — Sampling from different epochs

In this exercise, you will explore how the model's output changes across training epochs.

### Tasks:

1. **Select 5 checkpoints** spanning the training run (e.g., epochs 2, 10, 20, 40, and the final one)
2. **Sample molecules from each checkpoint** using a fixed temperature (e.g., `1.0`)
3. **Analyze patterns**, see notebook 6 for evaluations and apply them to the sampled molecules from the different checkpoints:
   - How does chemical diversity change across epochs?
   - Do later checkpoints show signs of overfitting (i.e., only generating molecules similar to the training set)?


**Hint:** Modify the code cell above to loop over different `epoch_idx` values and collect the results.
